In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
import requests
from dotenv import load_dotenv
import os
load_dotenv()

EXCHANGE_API_KEY = os.getenv("EXCHANGE_API_KEY")

In [2]:
@tool
def get_conversion_rate(base_currency: str, target_currency: str) -> float:
    """
    Get the conversion rate from base_currency to target_currency using an external API.
    """

    
    api_url = f"https://v6.exchangerate-api.com/v6/{EXCHANGE_API_KEY}/pair/{base_currency}/{target_currency}"
    response = requests.get(api_url)

    # print(response)
    return response.json()

In [3]:
# res = get_conversion_rate.invoke({"base_currency": "USD", "target_currency": "EUR"})
# print(res['conversion_rate'])

In [4]:
@tool
def convert_currency(amount: float, exchange_rate : float) -> float:
    """
    Convert an amount from one currency to another using the provided exchange rate.
    """
    return amount * exchange_rate

In [ ]:
model = ChatOllama(model = 'gemma4:e4b')
model_with_tools = model.bind_tools([convert_currency, get_conversion_rate])

messages = [SystemMessage(content="""You are a currency conversion assistant. 
    You MUST follow these two steps in exact order:
    1. FIRST, always use the `get_conversion_rate` tool to find the current exchange rate.
    2. SECOND, use the `convert_currency` tool to perform the final math."""),
    HumanMessage(content="What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR")
]

res = model_with_tools.invoke(messages)
messages.append(res)
tool_calls = res.tool_calls
print(res)

content='' additional_kwargs={} response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-10-01T14:58:59.9797218Z', 'done': True, 'done_reason': 'stop', 'total_duration': 43798762700, 'load_duration': 23618665100, 'prompt_eval_count': 246, 'prompt_eval_duration': 3635512000, 'eval_count': 206, 'eval_duration': 16529088000, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'} id='lc_run--01a0f7f9-27f1-7fa2-89d6-f1374190fe49-0' tool_calls=[{'name': 'get_conversion_rate', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': '816e9ce6-678f-4624-9c3c-846d27a70dd0', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 246, 'output_tokens': 206, 'total_tokens': 452}


In [6]:
for tool_call in tool_calls:
    if tool_call['name'] == "get_conversion_rate":
        res = get_conversion_rate.invoke(tool_call)
        messages.append(res)

    if tool_call['name'] == "convert_currency":
        res = convert_currency.invoke(tool_call['args'])
        messages.append(res)

In [7]:
print(messages)

[SystemMessage(content='You are a currency conversion assistant. \n    You MUST follow these two steps in exact order:\n    1. FIRST, always use the `get_conversion_rate` tool to find the current exchange rate.\n    2. SECOND, use the `convert_currency` tool to perform the final math.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-10-01T14:58:59.9797218Z', 'done': True, 'done_reason': 'stop', 'total_duration': 43798762700, 'load_duration': 23618665100, 'prompt_eval_count': 246, 'prompt_eval_duration': 3635512000, 'eval_count': 206, 'eval_duration': 16529088000, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--01a0f7f9-27f1-7fa2-89d6-f1374190fe49-0', tool_calls=[{'name': 'get_conv

In [8]:
res = model_with_tools.invoke(messages)
tool_calls = res.tool_calls
# print(tool_calls)
messages.append(res)
print(messages)

[SystemMessage(content='You are a currency conversion assistant. \n    You MUST follow these two steps in exact order:\n    1. FIRST, always use the `get_conversion_rate` tool to find the current exchange rate.\n    2. SECOND, use the `convert_currency` tool to perform the final math.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-10-01T14:58:59.9797218Z', 'done': True, 'done_reason': 'stop', 'total_duration': 43798762700, 'load_duration': 23618665100, 'prompt_eval_count': 246, 'prompt_eval_duration': 3635512000, 'eval_count': 206, 'eval_duration': 16529088000, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--01a0f7f9-27f1-7fa2-89d6-f1374190fe49-0', tool_calls=[{'name': 'get_conv

In [ ]:
for tool_call in tool_calls:
    if tool_call['name'] == "get_conversion_rate":
        res = get_conversion_rate.invoke(tool_call)
        messages.append(res)

    if tool_call['name'] == "convert_currency":
        res = convert_currency.invoke(tool_call)
        messages.append(res)

In [ ]:
response = model_with_tools.invoke(messages)
print(response.content)

The conversion factor from USD to INR is **96.0159**.

Based on this rate, 10 USD converts to **960.159 INR**.
